In [3]:
import arff
import pandas as pd

# загружаем данные
df = pd.read_parquet("c:/Personal/Science/Data/Binary_classification/GermanCredit.pq")
pd.set_option("display.max_columns", None)
df.head()

,Unnamed: 0,acc_status,acc_duration,credit_history,purpose,credit_amount,savings_acc,employment_time,installment_rate,gender,other_debtors,residence_time,property,age,installment_plans,housing,num_credits_at_bank,job,num_people_maintenance,phone,foreigner,target
0,600,dm_0_to_200,7,credit_paid_so_far,radio_or_tv,2329,dm_less_100,less_one,1,female,guarantor,1,real_estate,45,none,own,1,skilled_employee_or_official,1,no,yes,1
1,80,no_acc,24,credit_paid_so_far,radio_or_tv,5943,unknown_or_no_savings,less_one,1,female,none,1,car_or_other,44,none,own,2,skilled_employee_or_official,1,yes,yes,2
2,158,dm_0_to_200,24,credit_paid_so_far,car_new,3512,dm_100_to_500,four_to_seven,2,male,none,3,car_or_other,38,bank,own,2,skilled_employee_or_official,1,yes,yes,1
3,423,no_acc,21,delay_in_past_credit_payment,furniture_or_equipment,1591,dm_100_to_500,four_to_seven,4,male,none,3,real_estate,34,none,own,2,self_or_highly_qualified_employee,1,no,yes,1
4,747,dm_less_than_0,12,credit_paid_so_far,car_new,1274,dm_less_100,less_one,3,female,none,1,real_estate,37,none,own,1,unskilled_resident,1,no,yes,2


In [5]:
from typing import List, Any
import numpy as np

def clean_dataset(
    df: pd.DataFrame,
    cols_to_drop: List[Any] = None,
    missing_values: List[Any] = None
) -> pd.DataFrame:
    """
    Предобработка датафрейма с учетом пользовательских меток пропусков:
    1. Заполняет пропущенные значения
       - Модой для категориальных/текстовых признаков.
       - Средним значением для числовых признаков.
    2. Удаляет признаки, у которых одно значение занимает 100% строк.
    """
    df_cleaned = df.copy()
    n_rows = len(df_cleaned)
    
    if n_rows == 0:
        return df_cleaned

    # Задаем список меток для пропущенных значений по умолчанию, если не передан свой
    if missing_values is None:
        missing_values = ['', ' ', 'N/A', 'NA', 'null', 'nan', 'NaN', 'None', 'none', '?', '-']

    # 1. Приведение пользовательских пропусков к np.nan и их импутация
    for col in df_cleaned.columns:
        # Заменяем все кастомные метки пропусков на np.nan
        df_cleaned[col] = df_cleaned[col].replace(missing_values, np.nan)

        if df_cleaned[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_cleaned[col]):
                # а) Числовые переменные -> средним значением
                mean_val = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(mean_val)
            else:
                # б) Категориальные переменные -> модой
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])

    # 2. Проверка условий для удаления колонок
    for col in df_cleaned.columns:
        series = df_cleaned[col]
        max_value_count = series.value_counts().iloc[0]
        max_value_ratio = max_value_count / n_rows

        # Если одинаковых значений = 100% -> к удалению
        if max_value_ratio == 1:
            cols_to_drop.append(col)

    # Удаляем выявленные колонки
    if cols_to_drop:
        df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    return df_cleaned

df = clean_dataset(df, ["Unnamed: 0"])

In [7]:
df.isnull().sum()

acc_status                0
acc_duration              0
credit_history            0
purpose                   0
credit_amount             0
savings_acc               0
employment_time           0
installment_rate          0
gender                    0
other_debtors             0
residence_time            0
property                  0
age                       0
installment_plans         0
housing                   0
num_credits_at_bank       0
job                       0
num_people_maintenance    0
phone                     0
foreigner                 0
target                    0
dtype: int64

In [9]:
df['target'] = df['target'].map({1 : 1, 2 : 0 })
y = df["target"]
X = df.drop(["target"], axis=1)

In [13]:
import time
import category_encoders as ce
from skrub import SimilarityEncoder, MinHashEncoder
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from k_fold_encoder import KFoldTargetEncoder
from k_fold_glmm_encoder import KFoldGLMMEncoder

cat_cols = ['acc_status','credit_history','purpose','savings_acc',
                    'employment_time','gender','other_debtors', 'property',
                    'installment_plans','housing', 'foreigner', "phone", "job"]

encoders = {
    "MinHash": lambda: ColumnTransformer([(
                f"minhash_{col}",
                MinHashEncoder(),
                [col]
            )
            for col in cat_cols
        ],
        remainder="passthrough"
    ),
    "Similarity": lambda: ColumnTransformer([
        ("cat", SimilarityEncoder(), cat_cols)], 
        remainder="passthrough"),
   'Leave-one-out': lambda: ce.LeaveOneOutEncoder(cols=cat_cols),
   'K-fold':        lambda: KFoldTargetEncoder(cols=cat_cols),
   "CatBoost":      lambda: ce.CatBoostEncoder(cols=cat_cols),
   "CV-GLMM":       lambda: KFoldGLMMEncoder(cols=cat_cols),
   "WOE":           lambda: ce.WOEEncoder(cols=cat_cols),
   "JS":            lambda: ce.JamesSteinEncoder(cols=cat_cols),
   "MT":            lambda: ce.TargetEncoder(cols=cat_cols),
   'GLMM':          lambda: ce.GLMMEncoder(cols=cat_cols),
   'M-Estimate':    lambda: ce.MEstimateEncoder(cols=cat_cols),
   "OneHot":  lambda: ce.OneHotEncoder(cols=cat_cols),
   "Gray":    lambda: ce.GrayEncoder(cols=cat_cols),
   "BaseN":   lambda: ce.BaseNEncoder(cols=cat_cols),
   "Hashing": lambda: ce.HashingEncoder(cols=cat_cols,n_components=32),
   "Helmert": lambda: ce.HelmertEncoder(cols=cat_cols),
   "Sum":     lambda: ce.SumEncoder(cols=cat_cols),
   "BackwardDifference": lambda: ce.BackwardDifferenceEncoder(cols=cat_cols)
}

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():
    for encoder_name, encoder in encoders.items():
        
        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            enc_instance = encoder()  # Вызываем lambda, получая объект кодировщика
            X_train_encoded = enc_instance.fit_transform(X_train, y_train)
            X_test_encoded = enc_instance.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Empirical",
                    "dataset": "GermanCredit",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Empirical, GermanCredit, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_german_credit_empirical.csv", index=True)
print("Done!")

Empirical, GermanCredit, LogReg, MinHash, 1, 1, 0.7282142857142857, 0.8577460647643427, 0.735, 0.8203389830508474, 0.7806451612903226, 0.8642857142857143, 0.16306424140930176
Empirical, GermanCredit, LogReg, MinHash, 1, 2, 0.7740476190476191, 0.8701213631713443, 0.775, 0.8442906574394463, 0.8187919463087249, 0.8714285714285714, 0.11346101760864258
Empirical, GermanCredit, LogReg, MinHash, 1, 3, 0.8033333333333332, 0.9021831760840013, 0.76, 0.8356164383561644, 0.8026315789473685, 0.8714285714285714, 0.10953211784362793
Empirical, GermanCredit, LogReg, MinHash, 1, 4, 0.775952380952381, 0.8909865090659226, 0.745, 0.8282828282828283, 0.7834394904458599, 0.8785714285714286, 0.10475778579711914
Empirical, GermanCredit, LogReg, MinHash, 1, 5, 0.7947619047619048, 0.8827795988865677, 0.775, 0.845360824742268, 0.8145695364238411, 0.8785714285714286, 0.10595870018005371
Empirical, GermanCredit, LogReg, MinHash, 2, 1, 0.7476190476190476, 0.8825664510470947, 0.705, 0.7958477508650519, 0.77181208053

C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, GermanCredit, LightGBM, MinHash, 1, 2, 0.7903571428571428, 0.8685033695041621, 0.775, 0.8442906574394463, 0.8187919463087249, 0.8714285714285714, 0.35248756408691406


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, GermanCredit, LightGBM, MinHash, 1, 3, 0.7871428571428571, 0.8885118663514532, 0.785, 0.8561872909698997, 0.8050314465408805, 0.9142857142857143, 0.38696932792663574


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


... (85 outputs left out)

C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, GermanCredit, LightGBM, Similarity, 5, 4, 0.7090476190476191, 0.847530895230351, 0.705, 0.802675585284281, 0.7547169811320755, 0.8571428571428571, 0.23163747787475586


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, GermanCredit, LightGBM, Similarity, 5, 5, 0.7578571428571429, 0.851168194035393, 0.765, 0.8448844884488449, 0.7852760736196319, 0.9142857142857143, 0.23470330238342285
Empirical, GermanCredit, LightGBM, Leave-one-out, 1, 1, 0.4880357142857143, 0.6928972446437116, 0.7, 0.8235294117647058, 0.7, 1.0, 0.21848368644714355
Empirical, GermanCredit, LightGBM, Leave-one-out, 1, 2, 0.4311309523809524, 0.6592662910192719, 0.7, 0.8235294117647058, 0.7, 1.0, 0.2286207675933838
Empirical, GermanCredit, LightGBM, Leave-one-out, 1, 3, 0.4838095238095238, 0.6704390539018945, 0.7, 0.8235294117647058, 0.7, 1.0, 0.22845053672790527
Empirical, GermanCredit, LightGBM, Leave-one-out, 1, 4, 0.48642857142857143, 0.6742944107031925, 0.7, 0.8235294117647058, 0.7, 1.0, 0.20999646186828613
Empirical, GermanCredit, LightGBM, Leave-one-out, 1, 5, 0.5282738095238095, 0.7150093482937214, 0.7, 0.8235294117647058, 0.7, 1.0, 0.22625136375427246
Empirical, GermanCredit, LightGBM, Leave-one-out, 2, 1, 0.49267857